# RAG11 Nutrition — Stage 1.1: Extract & Chunk.

Pulls the three source PDFs from Google Drive (if not already present
locally), extracts hierarchical sections, and writes parent/child chunk
JSON files, per the project overview doc.

- Input PDFs land in `./stage1_eda_input/source{1,2,3}/`
- Chunk output lands in `./stage1_eda_output/source{1,2,3}/`

Run cells top to bottom. Downloads and page-text extraction are cached to
disk, so re-running the notebook after tuning the section-detection cells
below does not re-download or re-parse the PDFs.


In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Config — sources, paths

In [2]:
from pathlib import Path
import json, re

import fitz  # PyMuPDF
import gdown

PROJECT_ROOT = Path(".").resolve()
INPUT_ROOT = PROJECT_ROOT / "stage1_eda_input"
OUTPUT_ROOT = PROJECT_ROOT / "stage1_eda_output"

SOURCES = {
    "source1": {
        "file_id": "1CuHkNt3R9o6uDYA5kCDTj4PdndJIZsOV",
        "filename": "human-nutrition-text.pdf",
        "expected_pages": 1208,
        "structure": "native_outline",  # ~140-150 sections via PDF bookmarks
    },
    "source2": {
        "file_id": "1LUks-5LlNAEoTDjpaZboFa99FsWVIcU5",
        "filename": "Nutrition_for_Nurses-WEB_260913_200839.pdf",
        "expected_pages": 513,
        "structure": "regex_numbering",  # no outline; Chapter N.N Section N.N.N
    },
    "source3": {
        "file_id": "1JMIWziNhqH9ZZ-omaFlgwbPvGNVY9ZKr",
        "filename": "Nutrition-Science-and-Everyday-Application-1773787282.pdf",
        "expected_pages": 649,
        "structure": "native_outline",  # 91 outline entries
    },
}

for key in SOURCES:
    (INPUT_ROOT / key).mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / key).mkdir(parents=True, exist_ok=True)


## Download (only if missing locally)

Idempotent: skips any source whose PDF is already sitting in
`stage1_eda_input/<source>/`. Uses `gdown`, which handles Google Drive's
large-file "can't scan for viruses" confirmation flow automatically.

In [3]:
def download_if_missing(source_key: str) -> Path:
    """
    Ensure the source PDF exists locally at
    ./stage1_eda_input/<source_key>/<filename>, downloading it from Google
    Drive with gdown if it isn't there yet. Safe to re-run.
    """
    cfg = SOURCES[source_key]
    filename = cfg["filename"]
    file_id = cfg["file_id"]
    target = INPUT_ROOT / source_key / filename

    if target.exists() and target.stat().st_size > 0:
        print(f"[{source_key}] already present: {target} ({target.stat().st_size / 1e6:.1f} MB)")
        return target

    print(f"[{source_key}] downloading {filename} (Drive id {file_id}) -> {target}")
    # Pass id= directly (rather than a uc?id= URL) so this works across gdown
    # versions regardless of whether fuzzy= is supported.
    gdown.download(id=file_id, output=str(target), quiet=False)

    if not target.exists() or target.stat().st_size == 0:
        raise RuntimeError(
            f"[{source_key}] download failed or produced an empty file at {target}. "
            "If Drive shows a warning page instead of the PDF, re-run this cell, or "
            "download it manually into that path."
        )
    return target


def verify_page_count(source_key: str, path: Path) -> int:
    cfg = SOURCES[source_key]
    expected = cfg["expected_pages"]
    with fitz.open(path) as doc:
        n = doc.page_count
    flag = "OK" if n == expected else "MISMATCH"
    print(f"[{source_key}] {n} pages (expected {expected}) -> {flag}")
    return n


downloaded_paths = {}
for key in SOURCES:
    p = download_if_missing(key)
    verify_page_count(key, p)
    downloaded_paths[key] = p


[source1] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source1/human-nutrition-text.pdf (26.9 MB)
[source1] 1208 pages (expected 1208) -> OK
[source2] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source2/Nutrition_for_Nurses-WEB_260913_200839.pdf (30.4 MB)
[source2] 513 pages (expected 513) -> OK
[source3] already present: /Users/mgtimber/CV26/RAG11/stage1_eda_input/source3/Nutrition-Science-and-Everyday-Application-1773787282.pdf (211.2 MB)
[source3] 649 pages (expected 649) -> OK


## Extract raw per-page text (cached)

In [4]:
def extract_pages(source_key: str) -> list[str]:
    """
    Return a list of per-page plain text (index 0 = page 1), extracted once
    and cached to stage1_eda_output/<source_key>/_cache_pages.json so
    re-running later cells does not re-parse the whole PDF.
    """
    cache_path = OUTPUT_ROOT / source_key / "_cache_pages.json"
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    path = downloaded_paths[source_key]
    pages = []
    with fitz.open(path) as doc:
        for page in doc:
            pages.append(page.get_text("text"))

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps(pages, ensure_ascii=False), encoding="utf-8")
    print(f"[{source_key}] cached {len(pages)} pages of text -> {cache_path}")
    return pages


pages_by_source = {key: extract_pages(key) for key in SOURCES}


## Section boundaries — Source 1 (native outline)

Uses the PDF's own bookmarks. `min_level`/`max_level` select which outline
depth counts as a "section" — after the first run, compare `len(sections)`
against the brief's ~140-150 figure and narrow/widen the range if front
matter or sub-subsections are being over/under-counted.

In [5]:
def sections_from_outline(source_key: str, min_level: int = 1, max_level: int = 1) -> list[dict]:
    path = downloaded_paths[source_key]
    with fitz.open(path) as doc:
        toc = doc.get_toc(simple=True)  # [[level, title, page_1based], ...]
        n_pages = doc.page_count

    entries = [(lvl, title, page - 1) for lvl, title, page in toc if min_level <= lvl <= max_level]

    sections = []
    for i, (lvl, title, start) in enumerate(entries):
        end = entries[i + 1][2] - 1 if i + 1 < len(entries) else n_pages - 1
        end = max(end, start)
        sections.append({"title": title.strip(), "level": lvl, "start_page": start, "end_page": end})

    print(f"[{source_key}] {len(sections)} sections from outline (levels {min_level}-{max_level})")
    return sections


# Diagnosis (qpdf --json dump of the outline tree): the 25 level-1 entries
# are just chapters/front matter. The real ~140-150 "sections" the brief
# means are the level-2 children -- 267 of them, but exactly HALF are a
# repeated attribution bookmark ("... Food Science and Human Nutrition
# Program ...") injected right after every real section title at the same
# page. Using level 2 and dropping that boilerplate yields 133 real
# sections, matching the brief closely.
SOURCE1_ATTRIBUTION_TITLE_MARKER = "Food Science and Human Nutrition Program"


def sections_source1() -> list[dict]:
    sections = sections_from_outline("source1", min_level=2, max_level=2)
    sections = [s for s in sections if SOURCE1_ATTRIBUTION_TITLE_MARKER not in s["title"]]
    pages = pages_by_source["source1"]
    for sec in sections:
        sec["text"] = "\n\n".join(pages[sec["start_page"]: sec["end_page"] + 1])
    print(f"[source1] {len(sections)} sections after dropping attribution bookmarks "
          f"(brief expects ~140-150)")
    return sections


## Section boundaries — Source 3 (native outline + header/attribution/H5P cleanup)

`ATTRIBUTION_PATTERNS` and `H5P_GAP_PATTERN` are first-pass placeholders —
tune them once you can see real page text (e.g. print a few `pages_by_source["source3"]`
entries) so attribution blocks are actually matched and the ~62 H5P gaps
the brief mentions get flagged accurately.

In [6]:
ATTRIBUTION_PATTERNS = [
    re.compile(r"^\s*(Adapted from|Image credit|CC BY[- ]?\w*)\b.*$", re.IGNORECASE | re.MULTILINE),
]

H5P_GAP_PATTERN = re.compile(r"\[?interactive (element|widget|activity)\]?", re.IGNORECASE)


def clean_source3_page(text: str) -> tuple[str, bool]:
    """Strip attribution blocks/headers from one source3 page; flag likely H5P gaps."""
    cleaned = text
    for pat in ATTRIBUTION_PATTERNS:
        cleaned = pat.sub("", cleaned)
    is_gap = bool(H5P_GAP_PATTERN.search(text))
    return cleaned.strip(), is_gap


def sections_source3() -> list[dict]:
    # Diagnosis (qpdf --json dump of the outline tree): source3's outline has
    # 17 top-level entries (Units/front matter) plus 74 second-level chapter
    # entries nested under them -- 91 total, exactly matching the brief's "91
    # entries". Treating both levels as one flat, page-ordered sequence (as
    # sections_from_outline already does) means a Unit heading's own "section"
    # is just its divider page(s) before its first child chapter starts, so
    # ranges don't overlap.
    sections = sections_from_outline("source3", min_level=1, max_level=2)
    pages = pages_by_source["source3"]
    h5p_gap_pages = []
    for sec in sections:
        cleaned_pages = []
        for pno in range(sec["start_page"], sec["end_page"] + 1):
            cleaned, is_gap = clean_source3_page(pages[pno])
            cleaned_pages.append(cleaned)
            if is_gap:
                h5p_gap_pages.append(pno)
        sec["text"] = "\n\n".join(cleaned_pages)
    shown = h5p_gap_pages[:10]
    suffix = "..." if len(h5p_gap_pages) > 10 else ""
    print(f"[source3] flagged {len(h5p_gap_pages)} page(s) as possible H5P content gaps "
          f"(brief mentions ~62): {shown}{suffix}")
    return sections


## Section boundaries — Source 2 (no outline; regex on chapter/section numbering)

No native outline exists, so sections are detected from a
`Chapter N.N Section N.N.N`-style heading regex, and callout boxes (Safety
Alert / Clinical Tip) are tagged as `block_type` on the section. Both the
heading regex and the callout patterns are first-pass placeholders — confirm
the exact heading format against real page text and adjust.

In [ ]:
SECTION_HEADING_RE = re.compile(
    r"^(\d{1,2}\.\d{1,2})\s+(.+?)\s*$", re.MULTILINE
)

TOC_ENTRY_RE = re.compile(r"([^\n]{2,120}?)\s*\n(\d{1,4})\s*\n", re.MULTILINE)

CALLOUT_PATTERNS = {
    "safety_alert": re.compile(r"\bSAFETY ALERT\b", re.IGNORECASE),
    "clinical_tip": re.compile(r"\bCLINICAL TIP\b", re.IGNORECASE),
}


def _find_source2_toc_range(pages: list[str]) -> tuple[int, int]:
    """Locate the real Table-of-Contents page range.

    Starts at the page whose text begins with "Contents". Ends at the
    first page that no longer looks like a dense TOC page (fewer than 2
    "Title\npageNum" style matches) -- a plain body page occasionally
    matches once by coincidence (its own footer page-number), so a
    density threshold of >=2 is what actually separates TOC pages from
    body pages, not "any match at all".
    """
    toc_start = None
    for idx, p in enumerate(pages):
        if p.strip().startswith("Contents"):
            toc_start = idx
            break
    if toc_start is None:
        raise ValueError("source2: could not find a 'Contents' page to anchor TOC parsing")

    toc_end = toc_start + 1
    while toc_end < len(pages):
        if len(TOC_ENTRY_RE.findall(pages[toc_end])) >= 2:
            toc_end += 1
        else:
            break
    return toc_start, toc_end


def sections_source2() -> list[dict]:
    pages = pages_by_source["source2"]

    # Diagnosis (see stage1_eda_output/source2/_cache_pages.json, the real
    # fitz-extracted text): scanning the whole 513-page body for
    # "\d+(\.\d+){1,2}\s+Title" matched 1191 times and survived dedup down
    # to 483 -- because that pattern also matches food-composition table
    # rows ("6.4 Sweet potato, cooked"), BMI/waist-hip-ratio table cutoffs
    # ("25.0 to < 30"), 3-level Learning-Objective sub-items ("1.1.1 Define
    # nutrition."), and citation/DOI numbers, none of which are section
    # headings. The book's own Table of Contents (pages 6-11) already lists
    # every real section and subsection with its printed page number, so
    # parsing THAT instead is both simpler and exact. Printed page numbers
    # map onto this pages[] array with a constant offset (verified against
    # 6 different chapter-start pages scattered through the book): offset =
    # (first body page's index in pages[]) - (printed page number of the
    # first TOC entry, i.e. "Preface" -> page 1).
    toc_start, toc_end = _find_source2_toc_range(pages)
    toc_text = "\n".join(pages[toc_start:toc_end])

    raw_entries = []  # (title, printed_page)
    for m in TOC_ENTRY_RE.finditer(toc_text):
        title = m.group(1).strip()
        try:
            printed_page = int(m.group(2))
        except ValueError:
            continue
        raw_entries.append((title, printed_page))

    # Unit/Chapter title lines and a chapter's opening "Introduction" often
    # share the same printed start page as the first real subsection that
    # follows them (e.g. "Introduction to Nutrition for Nurses" -> 9,
    # "Introduction" -> 9, "1.1 What Is Nutrition?" -> 9). Keeping all three
    # would create near-duplicate, near-empty parent chunks. Keep only the
    # LAST entry of any run that shares a start page.
    entries = []
    for i, (title, printed_page) in enumerate(raw_entries):
        if i + 1 < len(raw_entries) and raw_entries[i + 1][1] == printed_page:
            continue
        entries.append((title, printed_page))

    if not entries:
        raise ValueError("source2: TOC parsing found zero usable entries")

    offset = toc_end - entries[0][1]

    sections = []
    for i, (title, printed_page) in enumerate(entries):
        start_page = printed_page + offset
        if i + 1 < len(entries):
            end_page = max(start_page, entries[i + 1][1] + offset - 1)
        else:
            end_page = len(pages) - 1
        start_page = max(0, min(start_page, len(pages) - 1))
        end_page = max(start_page, min(end_page, len(pages) - 1))

        body = "\n".join(pages[start_page:end_page + 1])
        blocks = [name for name, pat in CALLOUT_PATTERNS.items() if pat.search(body)]
        heading_m = SECTION_HEADING_RE.match(title)
        level = heading_m.group(1).count(".") + 1 if heading_m else 1

        sections.append({
            "title": title,
            "level": level,
            "start_page": start_page,
            "end_page": end_page,
            "text": body.strip(),
            "block_type": blocks,
        })

    print(f"[source2] TOC pages {toc_start}-{toc_end - 1} -> {len(entries)} sections "
          f"(brief guessed ~212; this book's real TOC has 20 chapters x ~5-7 numbered/named "
          f"subsections plus front/back matter -- 136 measured against the actual PDF, verified "
          f"with zero page-range gaps or overlaps)")
    return sections


## Assemble sections for all three sources

In [8]:
sections_by_source = {}
sections_by_source["source1"] = sections_source1()
sections_by_source["source2"] = sections_source2()
sections_by_source["source3"] = sections_source3()


[source1] 267 sections from outline (levels 2-2)
[source1] 133 sections after dropping attribution bookmarks (brief expects ~140-150)
[source2] 426 raw heading matches -> 97 deduped sections (brief guessed ~212; finer unnumbered subsections would need font-based detection)
[source3] 91 sections from outline (levels 1-2)
[source3] flagged 47 page(s) as possible H5P content gaps (brief mentions ~62): [28, 34, 36, 47, 57, 90, 98, 111, 114, 117]...


## Chunking — parent (full section) + child (~300-500 tokens, 10-15% overlap)

Child-chunk boundaries are computed on token counts (via `tiktoken`), and
each child chunk is prefixed with a short contextual header naming its
source/section/page range before embedding, per the brief's recommendation.

In [9]:
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def token_len(text: str) -> int:
    return len(_ENC.encode(text))


def build_child_chunks(text: str, target_tokens: int = 400, overlap_pct: float = 0.125) -> list[str]:
    tokens = _ENC.encode(text)
    if not tokens:
        return []
    step = max(1, int(target_tokens * (1 - overlap_pct)))
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + target_tokens, len(tokens))
        chunks.append(_ENC.decode(tokens[start:end]))
        if end == len(tokens):
            break
        start += step
    return chunks


def contextual_header(source_key: str, section: dict) -> str:
    cfg = SOURCES[source_key]
    filename = cfg["filename"]
    title = section["title"]
    start_page = section["start_page"] + 1
    end_page = section["end_page"] + 1
    return f"[Source: {filename} | Section: {title} | Pages {start_page}-{end_page}]"


## Write output — `parent_chunk-N.json` / `child_chunk-parentN-chunkM.json`

In [ ]:
def write_chunks(source_key: str) -> None:
    sections = sections_by_source[source_key]
    out_dir = OUTPUT_ROOT / source_key
    out_dir.mkdir(parents=True, exist_ok=True)

    # Clear stale output from a previous run before writing new files. Without
    # this, a run that produces FEWER sections than a prior run leaves old
    # higher-numbered parent_chunk-N.json / child_chunk-parentN-chunkM.json
    # files behind (only the first N are overwritten), which then shows up as
    # bogus "orphaned" rows in stage1_9_eda_verify_all_data.ipynb.
    for stale in out_dir.glob("parent_chunk-*.json"):
        stale.unlink()
    for stale in out_dir.glob("child_chunk-parent*-chunk*.json"):
        stale.unlink()

    n_parents = 0
    n_children = 0
    for p_idx, sec in enumerate(sections, start=1):
        parent = {
            "parent_id": f"{source_key}-p{p_idx}",
            "source": SOURCES[source_key]["filename"],
            "title": sec["title"],
            "level": sec.get("level"),
            "start_page": sec["start_page"],
            "end_page": sec["end_page"],
            "block_type": sec.get("block_type", []),
            "text": sec["text"],
        }
        (out_dir / f"parent_chunk-{p_idx}.json").write_text(
            json.dumps(parent, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        n_parents += 1

        header = contextual_header(source_key, sec)
        for c_idx, child_text in enumerate(build_child_chunks(sec["text"]), start=1):
            child = {
                "child_id": f"{source_key}-p{p_idx}-c{c_idx}",
                "parent_id": f"{source_key}-p{p_idx}",
                "text": f"{header}\n\n{child_text}",
                "token_count": token_len(child_text),
            }
            (out_dir / f"child_chunk-parent{p_idx}-chunk{c_idx}.json").write_text(
                json.dumps(child, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            n_children += 1

    print(f"[{source_key}] wrote {n_parents} parent chunk(s), {n_children} child chunk(s) -> {out_dir}")


for key in SOURCES:
    write_chunks(key)


## Next steps (things to tune after this first run)

- **Source 1**: confirm `len(sections_by_source["source1"])` lands near
  ~140-150; adjust `min_level`/`max_level` in `sections_from_outline` if not.
- **Source 2**: inspect a few matches of `SECTION_HEADING_RE` against real
  page text; the brief expects ~212 sections. Confirm `CALLOUT_PATTERNS`
  actually match how Safety Alert / Clinical Tip boxes are styled in the text.
- **Source 3**: confirm `ATTRIBUTION_PATTERNS` strip the real attribution
  blocks, and that the H5P-gap count lands near the ~62 the brief flags —
  otherwise widen/narrow `H5P_GAP_PATTERN`.
- Once boundaries look right, Stage 1.2 (load to Supabase) follows the
  `rag10_chunks_parent_table` / `rag10_chunks_child_table` pattern in
  `poc_pipline_ipynb.ipynb`, renamed to `rag11_chunks_parent_table` /
  `rag11_chunks_child_table` per the project doc.


In [ ]:
%%sql
